# COSC2793 Computational Machine Learning - Assignment 2

This notebook follows the assessment criteria (A-E) and mirrors the lab workflow.
Add short observations after each plot or experiment; be concise but justified.

**Assessment map (use these headings in your report):**
- A. Task and Dataset Understanding
- B. EDA and Data Handling
- C. Model Development and Performance Analysis
- D. Model Comparison
- E. Final Model Selection and Evaluation

In [ ]:
# Core imports and config
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score, classification_report, ConfusionMatrixDisplay
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.svm import SVC

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader

RANDOM_STATE = 0
np.random.seed(RANDOM_STATE)
torch.manual_seed(RANDOM_STATE)

In [ ]:
# Load data
train_path = './wildfire_cls_train_full.csv'
test_path = './wildfire_cls_test_features.csv'

df = pd.read_csv(train_path)
df_test = pd.read_csv(test_path)

print('Train shape:', df.shape)
print('Test shape:', df_test.shape)
df.head()

# A. Task and Dataset Understanding
**Task:** Multi-class classification to predict `fire_intensity` from wildfire observation features.

**Dataset characteristics:** Mixed numeric and categorical features (location, time, satellite, weather).
Potential challenges include missing values and class imbalance.

**Observation:** Add a short summary of the class distribution and missing values after running the next cell.

# B. EDA and Data Handling
We examine numeric correlations, feature-target relationships, and missingness before modeling.

**Observation:** Summarize any strong correlations or notable feature-target relationships.

In [ ]:
target_col = 'fire_intensity'
print('Target classes:', df[target_col].unique())
print('\nClass counts:')
print(df[target_col].value_counts())

plt.figure(figsize=(6, 4))
df[target_col].value_counts().plot(kind='bar')
plt.title('Class Distribution (fire_intensity)')
plt.xlabel('Class')
plt.ylabel('Count')
plt.grid(axis='y', alpha=0.3)
plt.show()

print('\nMissing values (top 10):')
print(df.isna().sum().sort_values(ascending=False).head(10))

# Numeric feature correlation with target (encoded)
df_numeric = df.select_dtypes(include=[np.number]).copy()
target_codes = df[target_col].astype('category').cat.codes
corrs = df_numeric.corrwith(target_codes).sort_values(key=np.abs, ascending=False)
print('\nTop numeric correlations with target:')
print(corrs.head(10))

# Correlation heatmap for numeric features
plt.figure(figsize=(8, 6))
plt.imshow(df_numeric.corr(), cmap='coolwarm', vmin=-1, vmax=1)
plt.title('Numeric Feature Correlation Heatmap')
plt.colorbar()
plt.tight_layout()
plt.show()

In [ ]:
# Split and identify column types
X = df.drop(columns=[target_col])
y = df[target_col]

X_train_val, X_test, y_train_val, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y
)
X_train, X_val, y_train, y_val = train_test_split(
    X_train_val, y_train_val, test_size=0.25, random_state=RANDOM_STATE, stratify=y_train_val
)

cat_cols = X_train.select_dtypes(include=['object', 'category']).columns.tolist()
num_cols = X_train.select_dtypes(include=[np.number]).columns.tolist()
print('Categorical cols:', cat_cols)
print('Numeric cols:', num_cols)

# Preprocessor for tree (no scaling)
preprocess_tree = ColumnTransformer(
    transformers=[
        ('num', Pipeline([('imputer', SimpleImputer(strategy='median'))]), num_cols),
        ('cat', Pipeline([
            ('imputer', SimpleImputer(strategy='most_frequent')),
            ('onehot', OneHotEncoder(handle_unknown='ignore'))
        ]), cat_cols)
    ]
)

# Preprocessor for SVM (scale numeric features)
preprocess_svm = ColumnTransformer(
    transformers=[
        ('num', Pipeline([
            ('imputer', SimpleImputer(strategy='median')),
            ('scaler', StandardScaler())
        ]), num_cols),
        ('cat', Pipeline([
            ('imputer', SimpleImputer(strategy='most_frequent')),
            ('onehot', OneHotEncoder(handle_unknown='ignore'))
        ]), cat_cols)
    ]
)

# Preprocessor for NN (dense output)
preprocess_nn = ColumnTransformer(
    transformers=[
        ('num', Pipeline([
            ('imputer', SimpleImputer(strategy='median')),
            ('scaler', StandardScaler())
        ]), num_cols),
        ('cat', Pipeline([
            ('imputer', SimpleImputer(strategy='most_frequent')),
            ('onehot', OneHotEncoder(handle_unknown='ignore', sparse=False))
        ]), cat_cols)
    ]
)

# C. Model Development and Performance Analysis
## Decision Tree
**Why a Decision Tree?**
- Pros: Interpretable, handles non-linear splits, fast to train.
- Cons: High variance, can overfit without pruning.

We sweep `max_depth` to study model complexity and use post-pruning with `ccp_alpha`.
**Observation:** Explain underfitting/overfitting trends and the chosen depth/alpha.

In [ ]:
# Decision Tree: depth sweep
depths = [2, 4, 6, 8, 10, 12, 15]
train_scores, val_scores = [], []

for d in depths:
    model = Pipeline([
        ('prep', preprocess_tree),
        ('clf', DecisionTreeClassifier(
            max_depth=d, class_weight='balanced', random_state=RANDOM_STATE
        ))
    ])
    model.fit(X_train, y_train)
    pred_train = model.predict(X_train)
    pred_val = model.predict(X_val)
    train_scores.append(f1_score(y_train, pred_train, average='macro'))
    val_scores.append(f1_score(y_val, pred_val, average='macro'))

plt.figure(figsize=(6, 4))
plt.plot(depths, train_scores, 'o-', label='train')
plt.plot(depths, val_scores, 'o-', label='val')
plt.xlabel('max_depth')
plt.ylabel('Macro F1')
plt.title('Decision Tree: Depth vs Macro F1')
plt.grid(alpha=0.3)
plt.legend()
plt.show()

# Fit best depth (replace with the best from your plot)
best_depth = 10
tree_model = Pipeline([
    ('prep', preprocess_tree),
    ('clf', DecisionTreeClassifier(
        max_depth=best_depth, class_weight='balanced', random_state=RANDOM_STATE
    ))
    ])
tree_model.fit(X_train, y_train)

val_pred = tree_model.predict(X_val)
print('Decision Tree Val Macro F1:', f1_score(y_val, val_pred, average='macro'))
print(classification_report(y_val, val_pred))

In [ ]:
# Visualize a shallow tree (readable)
tree_model_small = Pipeline([
    ('prep', preprocess_tree),
    ('clf', DecisionTreeClassifier(
        max_depth=3, class_weight='balanced', random_state=RANDOM_STATE
    ))
    ])
tree_model_small.fit(X_train, y_train)

feature_names = tree_model_small.named_steps['prep'].get_feature_names_out()
plt.figure(figsize=(16, 8))
plot_tree(
    tree_model_small.named_steps['clf'],
    feature_names=feature_names,
    class_names=tree_model_small.named_steps['clf'].classes_,
    filled=True,
    max_depth=3,
    fontsize=8
    )
plt.show()

# Post-pruning with cost complexity
tree_base = DecisionTreeClassifier(class_weight='balanced', random_state=RANDOM_STATE)
X_train_prep = preprocess_tree.fit_transform(X_train)
path = tree_base.cost_complexity_pruning_path(X_train_prep, y_train)
ccp_alphas = path.ccp_alphas

train_scores, val_scores = [], []
for alpha in ccp_alphas:
    clf = DecisionTreeClassifier(
        class_weight='balanced', random_state=RANDOM_STATE, ccp_alpha=alpha
    )
    clf.fit(X_train_prep, y_train)
    train_scores.append(f1_score(y_train, clf.predict(X_train_prep), average='macro'))
    val_scores.append(
        f1_score(y_val, clf.predict(preprocess_tree.transform(X_val)), average='macro')
    )

plt.figure(figsize=(6, 4))
plt.plot(ccp_alphas, train_scores, marker='o', label='train')
plt.plot(ccp_alphas, val_scores, marker='o', label='val')
plt.xlabel('ccp_alpha')
plt.ylabel('Macro F1')
plt.title('Decision Tree: Pruning Path')
plt.grid(alpha=0.3)
plt.legend()
plt.show()

## SVM
**Why SVM?**
- Pros: Strong margin-based classifier, works well in high dimensions.
- Cons: Sensitive to scaling and hyperparameters, can be slow on large datasets.

We compare linear vs RBF kernels and study sensitivity to `C` and `gamma`.
**Observation:** Explain how kernel choice and `C`/`gamma` change the train/val gap.

In [ ]:
# SVM: kernel and C sweep
C_values = [0.1, 1, 10]
kernels = ['linear', 'rbf']
results = []

for kernel in kernels:
    for C in C_values:
        model = Pipeline([
            ('prep', preprocess_svm),
            ('clf', SVC(kernel=kernel, C=C, gamma='scale'))
        ])
        model.fit(X_train, y_train)
        val_pred = model.predict(X_val)
        results.append({'kernel': kernel, 'C': C, 'val_f1': f1_score(y_val, val_pred, average='macro')})

res_df = pd.DataFrame(results)
print(res_df)

plt.figure(figsize=(6, 4))
for kernel in kernels:
    subset = res_df[res_df['kernel'] == kernel]
    plt.plot(subset['C'], subset['val_f1'], 'o-', label=kernel)
plt.xscale('log')
plt.xlabel('C')
plt.ylabel('Val Macro F1')
plt.title('SVM: Kernel and C')
plt.grid(alpha=0.3)
plt.legend()
plt.show()

# RBF gamma sensitivity
gamma_values = [0.01, 0.1, 1]
rbf_results = []

for C in [0.1, 1, 10]:
    for gamma in gamma_values:
        model = Pipeline([
            ('prep', preprocess_svm),
            ('clf', SVC(kernel='rbf', C=C, gamma=gamma))
        ])
        model.fit(X_train, y_train)
        val_pred = model.predict(X_val)
        rbf_results.append({'C': C, 'gamma': gamma, 'val_f1': f1_score(y_val, val_pred, average='macro')})

rbf_df = pd.DataFrame(rbf_results)
print(rbf_df)

plt.figure(figsize=(6, 4))
for gamma in gamma_values:
    subset = rbf_df[rbf_df['gamma'] == gamma]
    plt.plot(subset['C'], subset['val_f1'], 'o-', label=f'gamma={gamma}')
plt.xscale('log')
plt.xlabel('C')
plt.ylabel('Val Macro F1')
plt.title('SVM (RBF): C vs gamma')
plt.grid(alpha=0.3)
plt.legend()
plt.show()

# Fit best SVM (replace with your best C/gamma)
best_svm = Pipeline([
    ('prep', preprocess_svm),
    ('clf', SVC(kernel='rbf', C=1, gamma='scale'))
    ])
best_svm.fit(X_train, y_train)
svm_val_pred = best_svm.predict(X_val)
print('SVM Val Macro F1:', f1_score(y_val, svm_val_pred, average='macro'))
print(classification_report(y_val, svm_val_pred))

## Neural Network
**Why Neural Network?**
- Pros: Flexible function approximation, can model complex decision boundaries.
- Cons: Needs careful tuning, sensitive to learning rate and scaling.

We test different learning rates and analyze loss curves.
**Observation:** Identify the learning rate with stable convergence and discuss the train/val curves.

In [ ]:
# NN data prep
X_train_nn = preprocess_nn.fit_transform(X_train)
X_val_nn = preprocess_nn.transform(X_val)
X_test_nn = preprocess_nn.transform(X_test)

class_names = sorted(y_train.unique().tolist())
class_to_idx = {c: i for i, c in enumerate(class_names)}
y_train_nn = y_train.map(class_to_idx).values
y_val_nn = y_val.map(class_to_idx).values
y_test_nn = y_test.map(class_to_idx).values

class SimpleDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.FloatTensor(X)
        self.y = torch.LongTensor(y)
    def __len__(self):
        return len(self.X)
    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

batch_size = 64
train_loader = DataLoader(SimpleDataset(X_train_nn, y_train_nn), batch_size=batch_size, shuffle=True)
val_loader = DataLoader(SimpleDataset(X_val_nn, y_val_nn), batch_size=batch_size, shuffle=False)
test_loader = DataLoader(SimpleDataset(X_test_nn, y_test_nn), batch_size=batch_size, shuffle=False)

class MLP(nn.Module):
    def __init__(self, input_dim, hidden_dim, output_dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, output_dim)
        )
    def forward(self, x):
        return self.net(x)

def train_epoch(model, loader, criterion, optimizer, device):
    model.train()
    total_loss, correct, total = 0.0, 0, 0
    for Xb, yb in loader:
        Xb, yb = Xb.to(device), yb.to(device)
        optimizer.zero_grad()
        logits = model(Xb)
        loss = criterion(logits, yb)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
        preds = logits.argmax(dim=1)
        correct += (preds == yb).sum().item()
        total += yb.size(0)
    return total_loss / len(loader), correct / total

def eval_epoch(model, loader, criterion, device):
    model.eval()
    total_loss, correct, total = 0.0, 0, 0
    with torch.no_grad():
        for Xb, yb in loader:
            Xb, yb = Xb.to(device), yb.to(device)
            logits = model(Xb)
            loss = criterion(logits, yb)
            total_loss += loss.item()
            preds = logits.argmax(dim=1)
            correct += (preds == yb).sum().item()
            total += yb.size(0)
    return total_loss / len(loader), correct / total

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
input_dim = X_train_nn.shape[1]
output_dim = len(class_names)
hidden_dim = 64

# Learning rate experiment
lr_list = [0.001, 0.01, 0.05]
epochs = 50
histories = {}

for lr in lr_list:
    model = MLP(input_dim, hidden_dim, output_dim).to(device)
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.SGD(model.parameters(), lr=lr)
    history = {'train_loss': [], 'val_loss': [], 'train_acc': [], 'val_acc': []}
    for epoch in range(epochs):
        tr_loss, tr_acc = train_epoch(model, train_loader, criterion, optimizer, device)
        va_loss, va_acc = eval_epoch(model, val_loader, criterion, device)
        history['train_loss'].append(tr_loss)
        history['val_loss'].append(va_loss)
        history['train_acc'].append(tr_acc)
        history['val_acc'].append(va_acc)
    histories[lr] = history

plt.figure(figsize=(8, 4))
for lr in lr_list:
    plt.plot(histories[lr]['val_loss'], label=f'lr={lr}')
plt.xlabel('Epoch')
plt.ylabel('Val Loss')
plt.title('NN: Validation Loss for Different Learning Rates')
plt.grid(alpha=0.3)
plt.legend()
plt.show()

# Train final NN with chosen learning rate
best_lr = 0.01
epochs = 80
model = MLP(input_dim, hidden_dim, output_dim).to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.SGD(model.parameters(), lr=best_lr)

history = {'train_loss': [], 'val_loss': [], 'train_acc': [], 'val_acc': []}
for epoch in range(epochs):
    tr_loss, tr_acc = train_epoch(model, train_loader, criterion, optimizer, device)
    va_loss, va_acc = eval_epoch(model, val_loader, criterion, device)
    history['train_loss'].append(tr_loss)
    history['val_loss'].append(va_loss)
    history['train_acc'].append(tr_acc)
    history['val_acc'].append(va_acc)

plt.figure(figsize=(10, 4))
plt.subplot(1, 2, 1)
plt.plot(history['train_loss'], label='train')
plt.plot(history['val_loss'], label='val')
plt.title('NN Loss')
plt.xlabel('Epoch')
plt.legend()
plt.grid(alpha=0.3)
plt.subplot(1, 2, 2)
plt.plot(history['train_acc'], label='train')
plt.plot(history['val_acc'], label='val')
plt.title('NN Accuracy')
plt.xlabel('Epoch')
plt.legend()
plt.grid(alpha=0.3)
plt.show()

# NN validation report
model.eval()
all_preds, all_true = [], []
with torch.no_grad():
    for Xb, yb in val_loader:
        Xb = Xb.to(device)
        logits = model(Xb)
        preds = logits.argmax(dim=1).cpu().numpy()
        all_preds.extend(preds)
        all_true.extend(yb.numpy())

print('NN Val Macro F1:', f1_score(all_true, all_preds, average='macro'))
print(classification_report(all_true, all_preds, target_names=class_names))

# D. Model Comparison
Summarize best validation performance and stability for each model.
Discuss bias-variance tradeoff and hyperparameter sensitivity.

# E. Final Model Selection and Evaluation
Select the best model based on validation performance and stability, then evaluate on test.
Justify beyond performance (interpretability, robustness, compute cost).

In [ ]:
# Fill in with the best results from each model
comparison = pd.DataFrame([
    {'model': 'Decision Tree', 'val_macro_f1': None},
    {'model': 'SVM', 'val_macro_f1': None},
    {'model': 'Neural Network', 'val_macro_f1': None}
])
comparison

# Example: re-fit best model on train+val and evaluate on test
final_model = best_svm  # replace if you choose a different model
final_model.fit(X_train_val, y_train_val)
test_pred = final_model.predict(X_test)

print('Test Macro F1:', f1_score(y_test, test_pred, average='macro'))
print(classification_report(y_test, test_pred))
ConfusionMatrixDisplay.from_predictions(y_test, test_pred)
plt.show()

# Optional: generate predictions for the provided test features
final_test_preds = final_model.predict(df_test)
submission = pd.DataFrame({'fire_intensity': final_test_preds})
submission.to_csv('wildfire_predictions.csv', index=False)
submission.head()